# BNPL Payment Default & Collections Risk — Analysis

EDA on the cleaned Give Me Some Credit dataset (combined train + unlabeled test),
reframed with an installment / BNPL lending lens.

**Data:** `../data/processed/credit_data_clean.csv` and `../data/processed/credit_risk.db`  
**Policy:** feature EDA and SQL rates use labeled `source_flag = 'train'` only where outcomes are required.

## 1. Load & Inspect

In [1]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)

DATA_DIR = Path("../data")
CLEAN_PATH = DATA_DIR / "processed" / "credit_data_clean.csv"
DB_PATH = DATA_DIR / "processed" / "credit_risk.db"
RAW_TRAIN = DATA_DIR / "raw" / "cs-training.csv"
RAW_TEST = DATA_DIR / "raw" / "cs-test.csv"

df = pd.read_csv(CLEAN_PATH)
train = df.loc[df["source_flag"] == "train"].copy()
test = df.loc[df["source_flag"] == "test"].copy()

print(f"Combined shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Train (labeled): {len(train):,} | Test (unlabeled): {len(test):,}")
print(f"200K+ confirmed: {len(df) >= 200_000}")
df.head()

Combined shape: 251,503 rows × 17 columns
Train (labeled): 150,000 | Test (unlabeled): 101,503
200K+ confirmed: True


,id,serious_dlq_2yrs,revolving_utilization,age,times_30_59_days_late,debt_ratio,monthly_income,num_open_credit_lines,times_90_days_late,num_real_estate_loans,times_60_89_days_late,num_dependents,source_flag,late_payment_code_artifact,payment_to_income_ratio,prior_delinquency_count,revolving_utilization_bucket
0,1,1.0,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0,train,0,0.802982,2,High (>70%)
1,2,0.0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0,train,0,0.121876,0,High (>70%)
2,3,0.0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0,train,0,0.085113,2,Medium (30-70%)
3,4,0.0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0,train,0,0.036050,0,Low (<30%)
4,5,0.0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0,train,0,0.024926,1,High (>70%)


In [2]:
# Class balance — labeled train subset only
balance = train["serious_dlq_2yrs"].value_counts(dropna=False).rename(
    index={0.0: "Non-default (0)", 1.0: "Default (1)"}
)
balance_pct = train["serious_dlq_2yrs"].value_counts(normalize=True)
print("Class counts (train):")
print(balance.to_string())
print("\nClass share (train):")
print((balance_pct * 100).round(2).astype(str) + "%")

fig, ax = plt.subplots()
colors = ["#4C78A8", "#E45756"]
balance.plot(kind="bar", ax=ax, color=colors, rot=0)
ax.set_title("Class balance — train subset (serious_dlq_2yrs)")
ax.set_ylabel("Borrowers")
ax.set_xlabel("")
for i, (idx, val) in enumerate(balance.items()):
    pct = 100 * balance_pct.iloc[i]
    ax.text(i, val, f" {val:,}\n ({pct:.2f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, balance.max() * 1.15)
plt.tight_layout()
plt.show()

Class counts (train):
serious_dlq_2yrs
Non-default (0)    139974
Default (1)         10026

Class share (train):
serious_dlq_2yrs
0.0    93.32%
1.0     6.68%
Name: proportion, dtype: object


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/350703785.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Missingness before vs after cleaning (feature columns only;
# test-set NULL labels are expected and excluded from this comparison)
raw_train = pd.read_csv(RAW_TRAIN, index_col=0)
raw_test = pd.read_csv(RAW_TEST, index_col=0)
raw = pd.concat([raw_train, raw_test], ignore_index=True)

before_map = {
    "monthly_income": "MonthlyIncome",
    "num_dependents": "NumberOfDependents",
}
after_cols = ["monthly_income", "num_dependents", "payment_to_income_ratio"]

before_counts = pd.Series(
    {k: int(raw[before_map[k]].isna().sum()) for k in before_map},
    name="before",
)
# payment_to_income_ratio did not exist pre-clean
before_counts["payment_to_income_ratio"] = np.nan

after_counts = pd.Series(
    {c: int(df[c].isna().sum()) for c in after_cols},
    name="after",
)

miss = pd.concat([before_counts, after_counts], axis=1)
print("Missing value counts (combined population):")
display(miss)

plot_cols = ["monthly_income", "num_dependents"]
plot_df = miss.loc[plot_cols, ["before", "after"]].astype(float)

fig, ax = plt.subplots()
x = np.arange(len(plot_cols))
width = 0.35
ax.bar(x - width / 2, plot_df["before"], width, label="Before cleaning", color="#F58518")
ax.bar(x + width / 2, plot_df["after"], width, label="After cleaning", color="#54A24B")
ax.set_xticks(x)
ax.set_xticklabels(plot_cols)
ax.set_ylabel("Missing values")
ax.set_title("Missingness before vs after cleaning (combined train+test)")
ax.legend()
for i, col in enumerate(plot_cols):
    ax.text(i - width / 2, plot_df.loc[col, "before"], f"{int(plot_df.loc[col, 'before']):,}",
            ha="center", va="bottom", fontsize=8)
    ax.text(i + width / 2, plot_df.loc[col, "after"], f"{int(plot_df.loc[col, 'after']):,}",
            ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

print(
    f"Note: payment_to_income_ratio has {int(after_counts['payment_to_income_ratio']):,} NaNs "
    "after cleaning (monthly_income == 0); income/dependents are fully imputed."
)

Missing value counts (combined population):


,before,after
monthly_income,49834.0,0
num_dependents,6550.0,0
payment_to_income_ratio,NaN,2654


Note: payment_to_income_ratio has 2,654 NaNs after cleaning (monthly_income == 0); income/dependents are fully imputed.


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/2244889146.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. SQL Exploratory Results

Queries run against `credit_risk.db`, restricted to `source_flag = 'train'` (labeled rows only).

In [4]:
def run_sql(sql: str) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

q1 = run_sql(
    """
    SELECT
        COUNT(*) AS borrower_count,
        SUM(serious_dlq_2yrs) AS defaults,
        ROUND(100.0 * SUM(serious_dlq_2yrs) / COUNT(*), 2) AS default_rate_pct
    FROM borrowers
    WHERE source_flag = 'train'
    """
)
print("1. Overall default rate (train)")
display(q1)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Overall default rate"], q1["default_rate_pct"], color="#E45756")
ax.set_ylabel("Default rate (%)")
ax.set_title(f"Overall train default rate: {q1['default_rate_pct'].iloc[0]:.2f}%")
ax.set_ylim(0, max(10, q1["default_rate_pct"].iloc[0] * 1.5))
ax.text(0, q1["default_rate_pct"].iloc[0], f" {q1['default_rate_pct'].iloc[0]:.2f}%",
        ha="center", va="bottom")
plt.tight_layout()
plt.show()

1. Overall default rate (train)


,borrower_count,defaults,default_rate_pct
0,150000,10026,6.68


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/398228781.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
q2 = run_sql(
    """
    SELECT
        revolving_utilization_bucket,
        COUNT(*) AS borrower_count,
        ROUND(100.0 * SUM(serious_dlq_2yrs) / COUNT(*), 2) AS default_rate_pct
    FROM borrowers
    WHERE source_flag = 'train'
    GROUP BY revolving_utilization_bucket
    ORDER BY
        CASE revolving_utilization_bucket
            WHEN 'Low (<30%)' THEN 1
            WHEN 'Medium (30-70%)' THEN 2
            WHEN 'High (>70%)' THEN 3
            ELSE 4
        END
    """
)
print("2. Default rate by revolving_utilization_bucket")
display(q2)

fig, ax = plt.subplots()
sns.barplot(
    data=q2, x="revolving_utilization_bucket", y="default_rate_pct",
    hue="revolving_utilization_bucket", palette="Blues", legend=False, ax=ax,
)
ax.set_title("Default rate by revolving utilization bucket (train)")
ax.set_xlabel("")
ax.set_ylabel("Default rate (%)")
for i, row in q2.iterrows():
    ax.text(i, row["default_rate_pct"], f"{row['default_rate_pct']:.1f}%",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

2. Default rate by revolving_utilization_bucket


,revolving_utilization_bucket,borrower_count,default_rate_pct
0,Low (<30%),92882,2.22
1,Medium (30-70%),27170,7.40
2,High (>70%),29948,19.88


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/3514505479.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
q3 = run_sql(
    """
    SELECT
        CASE
            WHEN prior_delinquency_count = 0 THEN '0'
            WHEN prior_delinquency_count BETWEEN 1 AND 2 THEN '1-2'
            WHEN prior_delinquency_count BETWEEN 3 AND 5 THEN '3-5'
            ELSE '6+'
        END AS prior_delinquency_bucket,
        COUNT(*) AS borrower_count,
        ROUND(100.0 * SUM(serious_dlq_2yrs) / COUNT(*), 2) AS default_rate_pct
    FROM borrowers
    WHERE source_flag = 'train'
    GROUP BY prior_delinquency_bucket
    ORDER BY
        CASE
            WHEN prior_delinquency_bucket = '0' THEN 1
            WHEN prior_delinquency_bucket = '1-2' THEN 2
            WHEN prior_delinquency_bucket = '3-5' THEN 3
            ELSE 4
        END
    """
)
print("3. Default rate by prior_delinquency_count bucket")
display(q3)

fig, ax = plt.subplots()
sns.barplot(
    data=q3, x="prior_delinquency_bucket", y="default_rate_pct",
    hue="prior_delinquency_bucket", palette="Reds", legend=False, ax=ax,
)
ax.set_title("Default rate by prior delinquency count (train)")
ax.set_xlabel("Prior delinquency count")
ax.set_ylabel("Default rate (%)")
for i, row in q3.iterrows():
    ax.text(i, row["default_rate_pct"], f"{row['default_rate_pct']:.1f}%",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

3. Default rate by prior_delinquency_count bucket


,prior_delinquency_bucket,borrower_count,default_rate_pct
0,0,119906,2.84
1,1-2,23185,15.26
2,3-5,5419,39.97
3,6+,1490,61.21


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/3320962764.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
q4 = run_sql(
    """
    SELECT
        CAST(age / 10 * 10 AS INTEGER) AS age_decade_start,
        COUNT(*) AS borrower_count,
        ROUND(100.0 * SUM(serious_dlq_2yrs) / COUNT(*), 2) AS default_rate_pct
    FROM borrowers
    WHERE source_flag = 'train'
    GROUP BY age_decade_start
    ORDER BY age_decade_start
    """
)
print("4. Default rate by age decade")
display(q4)

fig, ax = plt.subplots()
sns.barplot(
    data=q4, x="age_decade_start", y="default_rate_pct",
    color="#72B7B2", ax=ax,
)
ax.set_title("Default rate by age decade (train)")
ax.set_xlabel("Age decade start")
ax.set_ylabel("Default rate (%)")
plt.tight_layout()
plt.show()

4. Default rate by age decade


,age_decade_start,borrower_count,default_rate_pct
0,20,8820,11.73
1,30,23183,10.07
2,40,34377,8.37
3,50,35302,6.45
4,60,28905,3.63
5,70,13601,2.43
6,80,5125,2.05
7,90,674,1.93
8,100,13,7.69


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/2271959640.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
q5 = run_sql(
    """
    SELECT
        serious_dlq_2yrs,
        COUNT(*) AS borrower_count,
        ROUND(AVG(payment_to_income_ratio), 4) AS avg_payment_to_income_ratio
    FROM borrowers
    WHERE source_flag = 'train'
    GROUP BY serious_dlq_2yrs
    """
)
print("5. Average payment_to_income_ratio by default status")
display(q5)

# Also show medians — means are skewed by DebtRatio absolute-dollar tail
pti_med = (
    train.groupby("serious_dlq_2yrs")["payment_to_income_ratio"]
    .median()
    .reset_index()
    .rename(columns={"payment_to_income_ratio": "median_pti"})
)
print("Median PTI (train) — more reliable than the mean:")
display(pti_med)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = q5["serious_dlq_2yrs"].map({0: "Non-default", 1: "Default"})
axes[0].bar(labels, q5["avg_payment_to_income_ratio"], color=["#4C78A8", "#E45756"])
axes[0].set_title("Mean PTI by default status")
axes[0].set_ylabel("Mean payment_to_income_ratio")

axes[1].bar(
    pti_med["serious_dlq_2yrs"].map({0: "Non-default", 1: "Default"}),
    pti_med["median_pti"],
    color=["#4C78A8", "#E45756"],
)
axes[1].set_title("Median PTI by default status")
axes[1].set_ylabel("Median payment_to_income_ratio")
plt.tight_layout()
plt.show()

5. Average payment_to_income_ratio by default status


,serious_dlq_2yrs,borrower_count,avg_payment_to_income_ratio
0,0,139974,309.1052
1,1,10026,247.6306


Median PTI (train) — more reliable than the mean:


,serious_dlq_2yrs,median_pti
0,0.0,0.358457
1,1.0,0.425918


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/2897333069.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Feature EDA

Distributions of the three engineered features by default status (**train subset only**),
plus a correlation heatmap against `serious_dlq_2yrs`.

In [9]:
# payment_to_income_ratio — clip to [0, 5] for readable density (matches ratio-scale values)
pti = train[["payment_to_income_ratio", "serious_dlq_2yrs"]].dropna().copy()
pti["pti_clip"] = pti["payment_to_income_ratio"].clip(0, 5)
pti["default_label"] = pti["serious_dlq_2yrs"].map({0: "Non-default", 1: "Default"})

fig, ax = plt.subplots()
sns.kdeplot(
    data=pti, x="pti_clip", hue="default_label", common_norm=False,
    fill=True, alpha=0.35, palette={"Non-default": "#4C78A8", "Default": "#E45756"}, ax=ax,
)
ax.set_title("payment_to_income_ratio by default status (clipped to [0, 5])")
ax.set_xlabel("payment_to_income_ratio (clipped)")
plt.tight_layout()
plt.show()

print(
    "Median PTI — Non-default:",
    round(pti.loc[pti["serious_dlq_2yrs"] == 0, "payment_to_income_ratio"].median(), 4),
    "| Default:",
    round(pti.loc[pti["serious_dlq_2yrs"] == 1, "payment_to_income_ratio"].median(), 4),
)

Median PTI — Non-default: 0.3585 | Default: 0.4259


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/4242665596.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# prior_delinquency_count — share of each count by default status
pdc = train.copy()
pdc["default_label"] = pdc["serious_dlq_2yrs"].map({0: "Non-default", 1: "Default"})
pdc["pdc_plot"] = pdc["prior_delinquency_count"].clip(upper=10)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(
    data=pdc, x="pdc_plot", hue="default_label", multiple="dodge", shrink=0.8,
    discrete=True, stat="probability", common_norm=False,
    palette={"Non-default": "#4C78A8", "Default": "#E45756"}, ax=ax,
)
ax.set_title("prior_delinquency_count by default status (train; counts clipped at 10)")
ax.set_xlabel("prior_delinquency_count")
ax.set_ylabel("Share within class")
plt.tight_layout()
plt.show()

print("Default rate by prior_delinquency bucket (from Section 2):")
display(q3)

Default rate by prior_delinquency bucket (from Section 2):


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/766129997.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,prior_delinquency_bucket,borrower_count,default_rate_pct
0,0,119906,2.84
1,1-2,23185,15.26
2,3-5,5419,39.97
3,6+,1490,61.21


In [11]:
# revolving_utilization_bucket — composition and default rate already shown in §2
util = (
    train.groupby(["revolving_utilization_bucket", "serious_dlq_2yrs"])
    .size()
    .unstack(fill_value=0)
)
util.columns = ["Non-default", "Default"]
util_pct = util.div(util.sum(axis=1), axis=0) * 100

order = ["Low (<30%)", "Medium (30-70%)", "High (>70%)"]
util = util.reindex(order)
util_pct = util_pct.reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
util.plot(kind="bar", stacked=True, ax=axes[0], color=["#4C78A8", "#E45756"], rot=15)
axes[0].set_title("Borrower counts by utilization bucket")
axes[0].set_xlabel("")
axes[0].set_ylabel("Borrowers")

sns.barplot(
    data=q2, x="revolving_utilization_bucket", y="default_rate_pct",
    hue="revolving_utilization_bucket", palette="Blues", legend=False, ax=axes[1],
)
axes[1].set_title("Default rate by utilization bucket")
axes[1].set_xlabel("")
axes[1].set_ylabel("Default rate (%)")
for i, row in q2.iterrows():
    axes[1].text(i, row["default_rate_pct"], f"{row['default_rate_pct']:.1f}%",
                 ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("Row % default within bucket:")
display(util_pct.round(2))

Row % default within bucket:


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/3944702301.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Non-default,Default
revolving_utilization_bucket,,
Low (<30%),97.78,2.22
Medium (30-70%),92.60,7.40
High (>70%),80.12,19.88


In [12]:
# Correlation heatmap: engineered + key raw features vs target
corr_cols = [
    "serious_dlq_2yrs",
    "payment_to_income_ratio",
    "prior_delinquency_count",
    "revolving_utilization",
    "debt_ratio",
    "monthly_income",
    "age",
    "times_30_59_days_late",
    "times_60_89_days_late",
    "times_90_days_late",
    "num_open_credit_lines",
    "num_dependents",
]
corr = train[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    vmin=-0.5, vmax=0.5, square=True, ax=ax,
)
ax.set_title("Correlation heatmap — engineered + key features (train)")
plt.tight_layout()
plt.show()

target_corr = (
    corr["serious_dlq_2yrs"]
    .drop("serious_dlq_2yrs")
    .sort_values(key=abs, ascending=False)
)
print("Correlation with serious_dlq_2yrs (sorted by |r|):")
display(target_corr.round(3).to_frame("corr_with_default"))

Correlation with serious_dlq_2yrs (sorted by |r|):


/var/folders/j5/0lm5qm4j4kb1sc_hkp_qrdhm0000gn/T/ipykernel_98360/3289946169.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,corr_with_default
prior_delinquency_count,0.389
times_90_days_late,0.312
revolving_utilization,0.281
times_30_59_days_late,0.271
times_60_89_days_late,0.266
age,-0.115
num_dependents,0.047
num_open_credit_lines,-0.030
monthly_income,-0.018
debt_ratio,-0.017


### Section 3 narrative

Among the three engineered features, **prior_delinquency_count** separates defaulters from non-defaulters most cleanly: default rates climb monotonically from ~2.8% with no prior lates to ~15% (1–2), ~40% (3–5), and ~61% (6+), the class-conditional histograms show defaulters heavily over-represented at higher counts, and it has the strongest correlation with `serious_dlq_2yrs` (~0.39). **revolving_utilization_bucket** is the next-strongest signal — Low / Medium / High move from ~2.2% to ~7.4% to ~19.9% default (nearly a 9× lift), with continuous `revolving_utilization` correlating at ~0.28. **payment_to_income_ratio** is the weakest separator: clipped densities overlap heavily, Pearson correlation is near zero (slightly negative, ~−0.02) because DebtRatio’s absolute-dollar right tail dominates the mean — which is *misleadingly lower* for defaulters — while only the median (≈0.43 vs ≈0.36) shows a mild elevation for defaults. For flagging high-risk borrowers before default, prioritize prior delinquency and revolving utilization; treat PTI as secondary affordability context, using median or winsorized views rather than raw means.
